***ADVANCED TEXT PREPROCESSING (INDO + EN + EMOJI + TYPO)***

# Preprocessing & Feature Engineering

Import Library & Load Data
---

In [1]:
# %pip install spellchecker

In [2]:
# %pip uninstall spellchecker -y

In [3]:
#%pip install pyspellchecker

In [4]:
import pandas as pd
import regex as re
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from spellchecker import SpellChecker
import emoji
from sklearn.model_selection import train_test_split

# Download stopwords
import nltk
nltk.download('stopwords')

ImportError: cannot import name 'SpellChecker' from 'spellchecker' (unknown location)

In [ ]:
df = pd.read_parquet('../data/cleaned/ulasan_dana_cleaned.parquet')
df.info()

Text Processing
---

Stack:
1. Hapus baris hanya emoji
2. Ekstrak emoji sebagai fitur
3. Bersihkan teks
4. Normalisasi kata non-baku
5. Hapus stopwords (ID + EN)
6. Spell Correction
7. Stemming Bahasa Indonesia
9. Label Sentiment

Hapus baris hanya emoji
***

In [ ]:
def contains_text(text):
    return bool(re.search(r'[a-zA-Z]', text))

df['has_text'] = df['ulasan'].apply(contains_text)
df_text = df[df['has_text']].drop(columns='has_text')

Ekstrak emoji sebagai fitur
***

In [ ]:
def extract_emoji(text):
    return "".join(c for c in text if c in emoji.EMOJI_DATA)

df_text['emojis'] = df_text['ulasan'].apply(extract_emoji)
df_text['emoji_count'] = df_text['emojis'].apply(len)

# Emoji sentiment score
emoji_sentiment = {
    '👍': 1, '🙏': 1, '🥰': 1, '😍': 1, '😊': 1, '😁': 1, '😘': 1, '🌟': 1, '💯': 1, '🤩': 1,
    '👎': -1, '😡': -1, '🤬': -1, '😢': -1, '😔': -1
}
df_text['emoji_sentiment_score'] = df_text['emojis'].apply(
    lambda s: sum(emoji_sentiment.get(e,0) for e in s)
)

Bersihkan teks
***

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#[\w-]+", "", text)
    text = emoji.replace_emoji(text, replace='')
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df_text['ulasan_clean'] = df_text['ulasan'].apply(clean_text)

Normalisasi kata non-baku
***

In [ ]:
normalisasi_dict = {
    'gak': 'tidak', 'ga': 'tidak', 'tdk': 'tidak', 'yg': 'yang',
    'dg': 'dengan', 'kalo': 'kalau', 'bgmn': 'bagaimana', 'tp': 'tapi'
}
def normalize_nonbaku(text):
    tokens = text.split()
    tokens = [normalisasi_dict.get(word, word) for word in tokens]
    return " ".join(tokens)
df_text['ulasan_clean'] = df_text['ulasan_clean'].apply(normalize_nonbaku)

Hapus stopwords (ID + EN)
***

In [ ]:
stop_words = set(stopwords.words('indonesian')).union(set(stopwords.words('english')))
def remove_stopwords(text):
    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words]
    return " ".join(tokens)
df_text['ulasan_clean'] = df_text['ulasan_clean'].apply(remove_stopwords)

Spell Correction
***

In [ ]:
from spellchecker import SpellChecker

spell_en = SpellChecker(language='en')

def correct_spelling(text):
    corrected_tokens = []
    for word in text.split():
        if word.lower() in spell_en:
            corrected_tokens.append(word)
        else:
            corrected_word = spell_en.correction(word)
            corrected_tokens.append(corrected_word if corrected_word else word)
    return " ".join(corrected_tokens)

df_text['ulasan_clean'] = df_text['ulasan_clean'].apply(correct_spelling)

Stemming Bahasa Indonesia
***

In [ ]:
factory = StemmerFactory()
stemmer = factory.create_stemmer()
df_text['ulasan_clean'] = df_text['ulasan_clean'].apply(lambda x: stemmer.stem(x))

Feature tambahan
***

In [ ]:
df_text['ulasan_len'] = df_text['ulasan_clean'].apply(len)
df_text['ulasan_word_count'] = df_text['ulasan_clean'].apply(lambda x: len(x.split()))
df_text['english_words_count'] = df_text['ulasan_clean'].apply(
    lambda x: sum(1 for w in x.split() if w in spell_en)
)

Label Sentiment
***

In [ ]:
def label_sentiment(rating):
    if rating >= 4:
        return 'positif'
    elif rating == 3:
        return 'netral'
    else:
        return 'negatif'

df_text['sentiment'] = df_text['rating'].apply(label_sentiment)

Split Data
---

In [ ]:
X = df_text['ulasan_clean']
y = df_text['sentiment']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Pipeline TF-IDF + XGBoost / RandomForest

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.metrics import classification_report

TF-IDF Vectorizer
---

In [ ]:
tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1,2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

RandomForest
---

In [ ]:
rf_model = RandomForestClassifier(n_estimators=200, random_state=42)
rf_model.fit(X_train_tfidf, y_train)
y_pred_rf = rf_model.predict(X_test_tfidf)
print("=== RandomForest Classification Report ===")
print(classification_report(y_test, y_pred_rf))

XGBoost
---

In [ ]:
xgb_model = xgb.XGBClassifier(
    n_estimators=300, max_depth=7, learning_rate=0.1, random_state=42, use_label_encoder=False, eval_metric='mlogloss'
)
xgb_model.fit(X_train_tfidf, y_train)
y_pred_xgb = xgb_model.predict(X_test_tfidf)
print("=== XGBoost Classification Report ===")
print(classification_report(y_test, y_pred_xgb))

# Pipeline BERT (Multilingual Indo–English

In [ ]:
from transformers import BertTokenizer, TFBertForSequenceClassification
import tensorflow as tf

Tokenizer & Model BERT Multilingual
---

In [ ]:
tokenizer = BertTokenizer.from_pretrained("bert-base-multilingual-cased")
max_len = 128

def encode_texts(texts):
    return tokenizer(
        list(texts),
        padding=True,
        truncation=True,
        max_length=max_len,
        return_tensors='tf'
    )

train_encodings = encode_texts(X_train)
test_encodings = encode_texts(X_test)

Convert labels ke angka
***

In [ ]:
label_map = {'negatif':0, 'netral':1, 'positif':2}
y_train_bert = y_train.map(label_map)
y_test_bert = y_test.map(label_map)

Model
---

In [ ]:
bert_model = TFBertForSequenceClassification.from_pretrained("bert-base-multilingual-cased", num_labels=3)

optimizer = tf.keras.optimizers.Adam(learning_rate=3e-5)
loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
bert_model.compile(optimizer=optimizer, loss=loss, metrics=['accuracy'])

Training (contoh 2 epoch untuk demo, bisa ditingkatkan)
***

In [ ]:
bert_model.fit(
    train_encodings['input_ids'],
    y_train_bert,
    validation_data=(test_encodings['input_ids'], y_test_bert),
    epochs=2,
    batch_size=16
)